In [0]:
### This is an end-to-end coding playbook that covers real-time industry specific coding questions based on certains scenarios

In [0]:
### Initializing the spark session 
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import*
spark=SparkSession.builder.appName("coding").getOrCreate()

In [0]:
#### QUESTION 2

### PROBLEM STATEMENT: Meridian is a 22-year-old IT services consultancy. The HR analytics team supports compensation reviews, attrition reporting, and the annual bonus calculation that goes to 8,500 employees. Last quarter's salary review surfaced inconsistencies. People were getting different ranks across reports for the same data. The Head of HR wants one source of truth and one set of definitions before the next cycle.

#### DATA COLLECTION: The Head of HR wants definitive employee rankings and bonus calculations. The CFO wants the rules expressed clearly enough that anyone can re-run them. The data team needs to demonstrate the right way to do rankings on the same dataset.

employee_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/employees.csv")

departments_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/departments.csv")

employee_salary=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/employee_salary.csv")

bonus_policy_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/bonus_policy.csv")

grades_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/grades.csv")

performance_df= spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/coding_playbook/2_meridianconsulting/raw_data/performance_reviews.csv")

In [0]:
### Task 1: Deliverable 1 – Data Quality Validation

### Before performing any analytical transformations or generating business reports, the HR datasets must undergo a comprehensive data quality validation process to ensure the accuracy and reliability of downstream calculations. 

# Analyze the employee, salary, and performance review datasets to identify records that violate business or data integrity rules. The validation process should detect duplicate employee IDs, duplicate email addresses, employees reporting to themselves, invalid manager IDs that do not exist in the employee master, future joining dates, null salaries, negative or zero salary values, null performance scores, and duplicate performance review records for the same employee and review year. 

# Generate separate reports for each type of data quality issue so that the HR Operations team can review and rectify the anomalies before the data is used for employee rankings, salary analysis, or bonus calculations. These validation reports will serve as an essential data quality checkpoint to ensure that all subsequent analyses are performed on clean, consistent, and trustworthy data.


In [0]:
## TASK 2: Deliverable 2 – Employee Hierarchy
### Using the employee master dataset, establish the reporting hierarchy by retrieving each employee's manager information through a self-join. Employees without managers should still appear in the final output.

return_orders= zenith_orders.alias("or").join(zenith_returns.alias("re"), on=f.col("or.order_id").cast("string")==f.col("re.order_id").cast("string"),how="inner").select("or.*",f.col("re.return_id"),f.col("re.return_date"))

return_orders_customers= return_orders.alias("or").join(zenith_customers.alias("ct"),on=f.col("or.customer_id").cast("string")==f.col("ct.customer_id").cast("string"),how="left").select("or.*",f.col("ct.customer_name"),f.col("ct.city"))

return_orders_customers_city= return_orders_customers.alias("or").join(zenith_city_tier.alias("ct"),on=f.col("or.city")==f.col("ct.city"),how="left").select("or.*",f.col("ct.city_tier"))

### FINAL SOLUTION
total_orders_placed= return_orders.count()
required_df= return_orders_customers_city.filter(f.col("city_tier")==f.lit("Tier-2"))
grouped_df= return_orders_customers_city.groupBy(f.col("product_category")).agg(f.count("*").alias("total_return_orders"))
grouped_df= grouped_df.withColumn("return_percentage",f.round((f.col("total_return_orders")/total_orders_placed),3))

grouped_df.show()

In [0]:
## TASK 3: Deliverable 3 – Salary Ranking within Grade
### Determine each employee's salary standing within their respective grade using the following ranking techniques: ROW_NUMBER(), RANK(), DENSE_RANK(). Aftwewards, Compare the differences in rankings where multiple employees earn identical salaries.


In [0]:
## TASK 4: Deliverable 4 – Top Three Performers by Department
### Identify the top three performing employees in every department based on their latest performance review. In situations where employees have identical performance scores, compare the behavior of different ranking functions.


In [0]:
### TASK 5: Deliverable 5 – Departments Requiring Salary Review
## Generate departmental salary statistics and identify departments meeting the following conditions:
# Average Salary greater than ₹120,000
# Employee Count greater than 25
# Total Salary Expense greater than ₹5,000,000
## Implement the filtering using aggregate functions and the HAVING clause.



In [0]:
### TASK 6: Deliverable 6 – Employee Bonus Calculation

## As part of the annual compensation review process, the HR department requires a standardized bonus calculation to ensure consistency across the organization. Using the latest salary and performance review data, calculate the bonus amount for every employee based on the predefined business rules. 

# Employees with a performance score of **95 or above** and a salary below ₹70,000 are eligible for a 25% bonus. 

# Employees with a performance score of **90 to 94** are eligible for a **20% bonus**, those scoring **80 to 89** receive a 15% bonus, employees scoring 70 to 79 receive a 10% bonus, and employees scoring below 70 receive a 5% bonus. 

# However, employees whose employment type is Contrac* or Intern, as well as employees whose employment status is Terminated, are not eligible for any bonus and should receive a bonus percentage of 0%.

# Calculate both the Bonus Percentage and the Bonus Amount for each eligible employee based on their latest salary. The final output should include the **Employee ID, Employee Name, Department, Employment Type, Latest Salary, Performance Score, Bonus Percentage, and Bonus Amount.


In [0]:
### TASK  7: Deliverable 7 – Employee Service Analysis and Latest Salary Determination

## Employees may have multiple salary records due to annual increments, promotions, or compensation revisions. As part of the data transformation process, determine the **latest salary** for each employee by selecting the most recent salary record based on the effective date using an appropriate window function. 

# Once the latest salary has been identified, calculate each employee's **years of service** using their joining date and the current date. Based on the calculated years of service, classify employees into the following service bands: 0–2 Years, 3–5 Years, 6–10 Years, and More than 10 Years. 

# The final output should include the Employee ID, Employee Name, Department, Joining Date, Latest Salary, Salary Effective Date, Years of Service, and Service Band. This curated dataset will be used by the HR team for compensation planning, workforce analytics, and employee retention initiatives.


In [0]:
### TASK  8: Deliverable 8 – Gold Layer Analytics

# Create a final gold-layer employee analytics table containing manager name, latest salary, performance score, salary rank, department rank, years of service, and bonus amount. This mirrors a real Medallion Architecture (Bronze → Silver → Gold) project in Azure Databricks.
